# 🏥 Notebook 3 — Hospital Recommender

**Project:** AI-Enabled Smart Emergency Response & Ambulance Coordination System
**Model role:** Given a patient + a candidate hospital, predict a *suitability score* in [0, 1].
The dispatcher UI shows the top-3 hospitals ranked by this score so the ambulance
goes to the *best-fit* hospital, not just the closest one.

### 🎯 Performance Target
- **R² ≥ 0.95** on score regression
- **Top-1 ranking accuracy ≥ 0.92** (correct best hospital out of 5 candidates)
- **Top-3 ranking accuracy ≥ 0.99**

### Score formula (the ground-truth)
```
score = w1 · specialty_match           (0/1)
      + w2 · bed_availability_match    (0–1, depends on patient need)
      + w3 · proximity                 (decays with distance)
      + w4 · low_er_wait               (decays with ER wait)
      + w5 · hospital_quality          (1–5 NABH-style rating, normalised)
      − penalty · is_diversion          (hospital at capacity)
```
The model must learn these non-linear interactions on its own.

## 1 · Setup & Imports

In [ ]:
# !pip install -q numpy pandas scikit-learn xgboost lightgbm catboost shap matplotlib seaborn joblib

import os, json, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, r2_score,
                             mean_squared_error, ndcg_score)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
print("Setup complete ✔")

## 2 · Synthetic Data Generation

Each row = one (emergency, candidate-hospital) pair.
We generate **scenarios** — one emergency tested against ~5 candidate hospitals — so we can
later evaluate ranking quality, not just regression error.

In [ ]:
PATIENT_TYPES = ["cardiac", "trauma", "stroke", "pediatric", "burns", "general"]
SPECIALTY_TO_BED = {
    "cardiac":   "icu",
    "trauma":    "trauma",
    "stroke":    "icu",
    "pediatric": "pediatric",
    "burns":     "burns",
    "general":   "general",
}

def proximity_score(distance_km):
    """1.0 at 0km decaying to ~0.1 at 25km."""
    return float(np.exp(-distance_km / 8))

def wait_score(wait_min):
    """1.0 at 0 min decaying to ~0.2 at 90 min."""
    return float(np.exp(-wait_min / 60))

def bed_score(available, total):
    if total == 0:
        return 0.0
    util = available / total
    # Reward >20% headroom; punish full
    return float(np.clip((util - 0.05) * 1.4, 0, 1))

def quality_score(rating):
    """Hospital NABH rating 1-5 → 0.2 - 1.0"""
    return rating / 5.0


def build_pair(scenario_id, patient_type, hosp):
    spec_match = int(patient_type in hosp["specialties"])
    bed_kind   = SPECIALTY_TO_BED[patient_type]
    avail = hosp[f"available_{bed_kind}"]
    total = hosp[f"total_{bed_kind}"]
    bed_avail_score = bed_score(avail, total)

    distance = float(np.random.uniform(0.5, 25.0))
    prox_s   = proximity_score(distance)
    wait_s   = wait_score(hosp["er_wait_minutes"])
    qual_s   = quality_score(hosp["quality_rating"])

    # Ground-truth score (what we ask the model to learn)
    score = (0.30 * spec_match +
             0.25 * bed_avail_score +
             0.20 * prox_s +
             0.15 * wait_s +
             0.10 * qual_s)
    if hosp["is_diversion"]:
        score *= 0.20         # heavy penalty
    score = float(np.clip(score + np.random.normal(0, 0.015), 0, 1))

    return {
        "scenario_id":         scenario_id,
        "patient_type_id":     PATIENT_TYPES.index(patient_type),
        "specialty_match":     spec_match,
        "distance_km":         round(distance, 2),
        "er_wait_minutes":     hosp["er_wait_minutes"],
        "is_diversion":        int(hosp["is_diversion"]),
        "quality_rating":      hosp["quality_rating"],
        # Bed availability per type
        "available_general":   hosp["available_general"],
        "total_general":       hosp["total_general"],
        "available_icu":       hosp["available_icu"],
        "total_icu":           hosp["total_icu"],
        "available_trauma":    hosp["available_trauma"],
        "total_trauma":        hosp["total_trauma"],
        "available_pediatric": hosp["available_pediatric"],
        "total_pediatric":     hosp["total_pediatric"],
        "available_burns":     hosp["available_burns"],
        "total_burns":         hosp["total_burns"],
        # Engineered: utilisation of relevant bed
        "relevant_bed_utilisation": (avail / total) if total > 0 else 0.0,
        "score":               score,
    }


def random_hospital():
    n_specialties = np.random.randint(2, 5)
    specs = list(np.random.choice(PATIENT_TYPES, size=n_specialties, replace=False))
    return {
        "specialties":      specs,
        "available_general":   int(np.random.randint(0, 80)),
        "total_general":       int(np.random.randint(60, 200)),
        "available_icu":       int(np.random.randint(0, 25)),
        "total_icu":           int(np.random.randint(20, 60)),
        "available_trauma":    int(np.random.randint(0, 12)),
        "total_trauma":        int(np.random.randint(8, 30)),
        "available_pediatric": int(np.random.randint(0, 20)),
        "total_pediatric":     int(np.random.randint(15, 40)),
        "available_burns":     int(np.random.randint(0, 8)),
        "total_burns":         int(np.random.randint(5, 15)),
        "er_wait_minutes":     int(np.random.exponential(35)),
        "is_diversion":        bool(np.random.rand() < 0.10),
        "quality_rating":      int(np.random.randint(2, 6)),
    }


def generate_recommender_dataset(n_scenarios=10_000, candidates_per=5):
    rows = []
    for sid in range(n_scenarios):
        ptype = np.random.choice(PATIENT_TYPES)
        for _ in range(candidates_per):
            rows.append(build_pair(sid, ptype, random_hospital()))
    return pd.DataFrame(rows)

df = generate_recommender_dataset()
print(f"Generated {len(df):,} candidate rows × {df.shape[1]} columns")
print(f"Number of scenarios: {df['scenario_id'].nunique():,}")
df.head()

## 3 · Exploratory Data Analysis

### 3.1 · Score distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(df["score"], bins=60, color="#2980b9", edgecolor="white")
ax.axvline(df["score"].mean(), color="red", ls="--",
           label=f"mean = {df['score'].mean():.3f}")
ax.set_title("Hospital recommendation score distribution")
ax.set_xlabel("score (0–1)"); ax.legend()
plt.tight_layout(); plt.show()

### 3.2 · Effect of specialty match & diversion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(x="specialty_match", y="score", data=df, ax=axes[0],
            palette=["#e74c3c","#27ae60"])
axes[0].set_title("Score: specialty match (1) vs no match (0)")
sns.boxplot(x="is_diversion", y="score", data=df, ax=axes[1],
            palette=["#27ae60","#e74c3c"])
axes[1].set_title("Score: hospital open (0) vs on diversion (1)")
plt.tight_layout(); plt.show()

### 3.3 · Score vs distance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sample = df.sample(3000, random_state=1)
sc = ax.scatter(sample["distance_km"], sample["score"],
                c=sample["specialty_match"], cmap="RdYlGn", s=10, alpha=0.6)
plt.colorbar(sc, ax=ax, label="specialty_match")
ax.set_xlabel("distance (km)"); ax.set_ylabel("score")
ax.set_title("Score decays with distance; specialty match shifts the band up")
plt.tight_layout(); plt.show()

### 3.4 · Bed utilisation impact

In [ ]:
df["bed_util_bin"] = pd.cut(df["relevant_bed_utilisation"],
                            bins=[-0.01, 0.10, 0.30, 0.60, 1.0],
                            labels=["≤10%","10-30%","30-60%","60-100%"])
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(x="bed_util_bin", y="score", data=df, ax=ax, palette="Greens")
ax.set_title("Score vs availability of the relevant bed type")
ax.set_xlabel("Available / Total of relevant bed type")
plt.tight_layout(); plt.show()
df.drop(columns=["bed_util_bin"], inplace=True)

### 3.5 · Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson correlation matrix")
plt.tight_layout(); plt.show()

## 4 · Train / Test Split

We split **by `scenario_id`** so that all 5 candidates of a scenario stay in the same fold.
This lets us compute true ranking metrics on the test set.

In [ ]:
TARGET = "score"
GROUP  = "scenario_id"
feature_cols = [c for c in df.columns if c not in (TARGET, GROUP)]

scenario_ids = df[GROUP].unique()
train_ids, test_ids = train_test_split(scenario_ids, test_size=0.20,
                                       random_state=RANDOM_STATE)
df_train = df[df[GROUP].isin(train_ids)].reset_index(drop=True)
df_test  = df[df[GROUP].isin(test_ids)].reset_index(drop=True)
print(f"Train: {df_train.shape}  Test: {df_test.shape}")

X_train = df_train[feature_cols].values
y_train = df_train[TARGET].values
X_test  = df_test[feature_cols].values
y_test  = df_test[TARGET].values

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

## 5 · Model bake-off

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te, do_cv=False):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    metrics = dict(
        MAE  = mean_absolute_error(y_te, preds),
        RMSE = np.sqrt(mean_squared_error(y_te, preds)),
        R2   = r2_score(y_te, preds))
    if do_cv:
        cv_r2 = cross_val_score(model, X_tr, y_tr,
                                cv=KFold(3, shuffle=True, random_state=RANDOM_STATE),
                                scoring="r2", n_jobs=-1).mean()
        metrics["CV_R2"] = cv_r2
    results[name] = {"model": model, "preds": preds, **metrics}
    extra = f"  CV-R²={metrics.get('CV_R2',0):.4f}" if do_cv else ""
    print(f"{name:18s}  MAE={metrics['MAE']:.5f}  RMSE={metrics['RMSE']:.5f}  "
          f"R²={metrics['R2']:.4f}{extra}")
    return model

### 5.1 · Linear baseline

In [ ]:
evaluate("Ridge", Ridge(alpha=1.0, random_state=RANDOM_STATE),
         X_train_s, y_train, X_test_s, y_test)

### 5.2 · Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=400, max_depth=18,
                           min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)
evaluate("RandomForest", rf, X_train_s, y_train, X_test_s, y_test)

### 5.3 · XGBoost (tuned)

In [ ]:
xgb = XGBRegressor(
    n_estimators=900, max_depth=8, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=2, reg_alpha=0.1, reg_lambda=1.2,
    objective="reg:squarederror", tree_method="hist",
    random_state=RANDOM_STATE, n_jobs=-1)
evaluate("XGBoost", xgb, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 5.4 · LightGBM

In [ ]:
lgb = LGBMRegressor(
    n_estimators=900, num_leaves=63, max_depth=-1,
    learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=0.5,
    objective="regression", random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1)
evaluate("LightGBM", lgb, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 5.5 · CatBoost

In [ ]:
cat = CatBoostRegressor(
    iterations=900, depth=8, learning_rate=0.05,
    l2_leaf_reg=3.0, loss_function="RMSE",
    random_seed=RANDOM_STATE, verbose=False)
evaluate("CatBoost", cat, X_train_s, y_train, X_test_s, y_test, do_cv=True)

### 5.6 · Ensemble (averaged)

In [ ]:
preds_avg = (results["XGBoost"]["preds"]
           + results["LightGBM"]["preds"]
           + results["CatBoost"]["preds"]) / 3
metrics = dict(
    MAE  = mean_absolute_error(y_test, preds_avg),
    RMSE = np.sqrt(mean_squared_error(y_test, preds_avg)),
    R2   = r2_score(y_test, preds_avg))
results["EnsembleAvg(XGB+LGB+Cat)"] = {"model":"avg", "preds":preds_avg, **metrics}
print(f"EnsembleAvg          MAE={metrics['MAE']:.5f}  RMSE={metrics['RMSE']:.5f}  "
      f"R²={metrics['R2']:.4f}")

## 6 · Leaderboard

In [ ]:
leaderboard = pd.DataFrame([
    {"model": k, "MAE": v["MAE"], "RMSE": v["RMSE"], "R2": v["R2"]}
    for k, v in results.items()
]).sort_values("R2", ascending=False).reset_index(drop=True)
display(leaderboard.round(5))

best_name = leaderboard.iloc[0]["model"]
print(f"\n🏆 Best regression model: {best_name}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
order = leaderboard.sort_values("R2")
ax.barh(order["model"], order["R2"], color="#27ae60")
ax.axvline(0.95, ls="--", color="red", label="0.95 target")
ax.set_xlim(0.5, 1.0); ax.set_xlabel("R²"); ax.legend()
ax.set_title("Hospital recommender R² leaderboard")
plt.tight_layout(); plt.show()

## 7 · Ranking metrics — the *real* business KPI

For each scenario in the test set, ask:
*"Did the model rank the hospitals in the same order the ground-truth would?"*

We measure **Top-1 accuracy**, **Top-3 accuracy**, and **NDCG@5**.

In [ ]:
def ranking_metrics(df_test, predictions):
    df_t = df_test.copy()
    df_t["pred"] = predictions
    top1, top3, ndcg_list = [], [], []
    for sid, grp in df_t.groupby(GROUP):
        true_order = grp.sort_values("score", ascending=False).index.tolist()
        pred_order = grp.sort_values("pred",  ascending=False).index.tolist()
        top1.append(int(true_order[0] == pred_order[0]))
        top3.append(int(true_order[0] in pred_order[:3]))
        ndcg_list.append(ndcg_score([grp["score"].values], [grp["pred"].values]))
    return {
        "top1_accuracy": float(np.mean(top1)),
        "top3_accuracy": float(np.mean(top3)),
        "ndcg@5":        float(np.mean(ndcg_list)),
    }

best_preds = results[best_name]["preds"]
ranking = ranking_metrics(df_test, best_preds)
for k, v in ranking.items():
    print(f"{k:18s}: {v:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
keys, vals = list(ranking.keys()), list(ranking.values())
bars = ax.bar(keys, vals, color=["#27ae60","#16a085","#2980b9"])
ax.axhline(0.92, ls="--", color="red", alpha=0.6, label="Top-1 target=0.92")
ax.set_ylim(0, 1.05); ax.set_ylabel("score")
ax.set_title("Ranking quality of the best model")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

## 8 · Diagnostics on the winner

### 8.1 · Predicted vs Actual

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, best_preds, s=4, alpha=0.35, color="#2980b9")
ax.plot([0,1],[0,1], "r--", lw=2)
ax.set_xlabel("Actual score"); ax.set_ylabel("Predicted score")
ax.set_title(f"{best_name}: predicted vs actual")
plt.tight_layout(); plt.show()

### 8.2 · Residuals

In [ ]:
residuals = y_test - best_preds
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].scatter(best_preds, residuals, s=4, alpha=0.3, color="#8e44ad")
axes[0].axhline(0, color="red"); axes[0].set_title("Residuals vs predicted")
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("residual")

axes[1].hist(residuals, bins=80, color="#8e44ad", edgecolor="white")
axes[1].set_title("Residual distribution"); axes[1].axvline(0, color="red")
plt.tight_layout(); plt.show()

### 8.3 · Feature importance

In [ ]:
imp_model = results["XGBoost"]["model"]
imp = pd.Series(imp_model.feature_importances_, index=feature_cols).sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
imp.plot.barh(ax=ax, color="#16a085")
ax.set_title("XGBoost feature importance — what drives a good hospital match?")
plt.tight_layout(); plt.show()

### 8.4 · SHAP global

In [ ]:
import shap
try:
    explainer  = shap.TreeExplainer(imp_model)
    shap_vals  = explainer.shap_values(X_test_s[:1000])
    shap.summary_plot(shap_vals, X_test_s[:1000],
                      feature_names=feature_cols, show=True)
except Exception as e:
    print("SHAP skipped:", e)

## 9 · Run the recommender end-to-end on a fresh scenario

In [ ]:
def rank_hospitals(patient_type, candidate_hospitals, model_name="XGBoost"):
    m = results[model_name]["model"]
    rows = [build_pair(scenario_id=-1, patient_type=patient_type, hosp=h)
            for h in candidate_hospitals]
    df_c = pd.DataFrame(rows)
    X = scaler.transform(df_c[feature_cols].values)
    df_c["predicted_score"] = m.predict(X)
    return (df_c[["predicted_score","score","specialty_match","distance_km",
                  "er_wait_minutes","is_diversion","quality_rating"]]
              .sort_values("predicted_score", ascending=False)
              .reset_index(drop=True))

candidate_hospitals = [random_hospital() for _ in range(5)]
print("Patient type: cardiac\n")
display(rank_hospitals("cardiac", candidate_hospitals).round(3))

## 10 · Persist artefacts

In [ ]:
joblib.dump(results["XGBoost"]["model"], "models/hospital_recommender.pkl")
joblib.dump(scaler,                      "models/hospital_scaler.pkl")
joblib.dump(feature_cols,                "models/hospital_feature_cols.pkl")

final_metrics = {
    "best_model": best_name,
    "R2":   round(results[best_name]["R2"], 4),
    "MAE":  round(results[best_name]["MAE"], 5),
    "ranking": {k: round(v,4) for k,v in ranking.items()},
}
with open("reports/hospital_recommender_report.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print(json.dumps(final_metrics, indent=2))
print("\n✅ Hospital recommender saved.")

## 11 · Summary

| Metric | Target | Achieved |
|--------|--------|----------|
| R² (regression)      | ≥ 0.95 | _see leaderboard_ |
| Top-1 ranking acc    | ≥ 0.92 | _see ranking chart_ |
| Top-3 ranking acc    | ≥ 0.99 | _see ranking chart_ |

**Why this works:**
- Engineered `relevant_bed_utilisation` exposes the most important non-linear interaction (patient type × bed availability).
- Splitting by `scenario_id` ensures honest ranking evaluation.
- Boosters dominate Ridge by a wide margin → the score function is genuinely non-linear.

→ Continue to **Notebook 4 — Traffic Predictor**.